Update from previous model:
- create batch on-the-fly -> use yield function
- binary classification with cross entropy
  - custom the label

In [1]:
import os

import pandas as pd
import polars as pl
import itertools
import numpy as np

import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score, roc_curve, auc, precision_recall_fscore_support, confusion_matrix, classification_report

In [2]:
WORKING_DIR = '/group/pmc021/amunif/epi-thesis/workflow/09_Learning_to_Rank'
DATASET_DIR = '/group/pmc021/amunif/epi-thesis/workflow/08_HepG2'

In [3]:
def get_device():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    
    if device.type == "cuda":
        print(f"Current CUDA device: {torch.cuda.current_device()}")
        print(f"CUDA device name: {torch.cuda.get_device_name(0)}")
    
    return device

In [4]:
device = get_device()
print(device)

Using device: cuda
Current CUDA device: 0
CUDA device name: Tesla P100-SXM2-16GB
cuda


In [5]:
# Load dataset
gene_pl = pl.read_parquet(os.path.join(DATASET_DIR, 'dataset', 'gene_w_label_value_1.parquet'))
gene_pl

gene_id,H3K4me3,H3K4me3_count,H3K9ac,H3K9ac_count,H3K9me3,H3K9me3_count,H3K27ac,H3K27ac_count,H3K27me3,H3K27me3_count,value_1,label
str,list[f64],i64,list[f64],i64,list[f64],i64,list[f64],i64,list[f64],i64,f64,i32
"""XLOC_000001""","[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,0.0,0
"""XLOC_000003""","[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,0.0,0
"""XLOC_000006""","[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,0.0888452,0
"""XLOC_000007""","[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,4.04743,1
"""XLOC_000008""","[0.0, 0.0, … 0.0]",6,"[0.0, 0.0, … 0.0]",3,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",2,"[0.0, 0.0, … 0.0]",0,26.7934,1
…,…,…,…,…,…,…,…,…,…,…,…,…
"""XLOC_030009""","[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,0.0,0
"""XLOC_030012""","[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,0.0,0
"""XLOC_030014""","[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,0.0,0


In [6]:
marker_list = ['H3K4me3', 'H3K9ac', 'H3K9me3', 'H3K27ac', 'H3K27me3']

gene_features = gene_pl.with_columns(
    pl.concat_list(marker_list).alias("combined_features")
)

In [7]:
gene_features

gene_id,H3K4me3,H3K4me3_count,H3K9ac,H3K9ac_count,H3K9me3,H3K9me3_count,H3K27ac,H3K27ac_count,H3K27me3,H3K27me3_count,value_1,label,combined_features
str,list[f64],i64,list[f64],i64,list[f64],i64,list[f64],i64,list[f64],i64,f64,i32,list[f64]
"""XLOC_000001""","[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,0.0,0,"[0.0, 0.0, … 0.0]"
"""XLOC_000003""","[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,0.0,0,"[0.0, 0.0, … 0.0]"
"""XLOC_000006""","[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,0.0888452,0,"[0.0, 0.0, … 0.0]"
"""XLOC_000007""","[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,4.04743,1,"[0.0, 0.0, … 0.0]"
"""XLOC_000008""","[0.0, 0.0, … 0.0]",6,"[0.0, 0.0, … 0.0]",3,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",2,"[0.0, 0.0, … 0.0]",0,26.7934,1,"[0.0, 0.0, … 0.0]"
…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""XLOC_030009""","[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,0.0,0,"[0.0, 0.0, … 0.0]"
"""XLOC_030012""","[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,0.0,0,"[0.0, 0.0, … 0.0]"
"""XLOC_030014""","[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,"[0.0, 0.0, … 0.0]",0,0.0,0,"[0.0, 0.0, … 0.0]"


In [8]:
# Select only the gene_id and the value_1
gene_id_val = gene_features.select(["gene_id", "value_1"])
gene_id_val

gene_id,value_1
str,f64
"""XLOC_000001""",0.0
"""XLOC_000003""",0.0
"""XLOC_000006""",0.0888452
"""XLOC_000007""",4.04743
"""XLOC_000008""",26.7934
…,…
"""XLOC_030009""",0.0
"""XLOC_030012""",0.0
"""XLOC_030014""",0.0


In [9]:
# Doing the permutation using cross join
permuted_df = gene_id_val.join(gene_id_val, how="cross")

In [10]:
# Create the label
permuted_df = permuted_df.with_columns(
    pl.when(pl.col("value_1") > pl.col("value_1_right")).then(1)
      .otherwise(0)
      .alias("label")
)

In [11]:
permuted_df

gene_id,value_1,gene_id_right,value_1_right,label
str,f64,str,f64,i32
"""XLOC_000001""",0.0,"""XLOC_000001""",0.0,0
"""XLOC_000001""",0.0,"""XLOC_000003""",0.0,0
"""XLOC_000001""",0.0,"""XLOC_000006""",0.0888452,0
"""XLOC_000001""",0.0,"""XLOC_000007""",4.04743,0
"""XLOC_000001""",0.0,"""XLOC_000008""",26.7934,0
…,…,…,…,…
"""XLOC_030018""",0.0,"""XLOC_030009""",0.0,0
"""XLOC_030018""",0.0,"""XLOC_030012""",0.0,0
"""XLOC_030018""",0.0,"""XLOC_030014""",0.0,0


In [12]:
# Check the label distribution
permuted_df.group_by("label").len()

label,len
i32,u32
1,232386241
0,258413475


In [13]:
# Take the sample
sample_df = permuted_df[:100_000]

In [14]:
sample_df.group_by("label").len()

label,len
i32,u32
0,69411
1,30589


In [15]:
sample_np = sample_df.to_numpy()

In [16]:
data_df = gene_features.select(["gene_id", "combined_features"])
data_df

gene_id,combined_features
str,list[f64]
"""XLOC_000001""","[0.0, 0.0, … 0.0]"
"""XLOC_000003""","[0.0, 0.0, … 0.0]"
"""XLOC_000006""","[0.0, 0.0, … 0.0]"
"""XLOC_000007""","[0.0, 0.0, … 0.0]"
"""XLOC_000008""","[0.0, 0.0, … 0.0]"
…,…
"""XLOC_030009""","[0.0, 0.0, … 0.0]"
"""XLOC_030012""","[0.0, 0.0, … 0.0]"
"""XLOC_030014""","[0.0, 0.0, … 0.0]"


In [17]:
data_np = data_df.to_numpy()
data_np

array([['XLOC_000001', array([0., 0., 0., ..., 0., 0., 0.])],
       ['XLOC_000003', array([0., 0., 0., ..., 0., 0., 0.])],
       ['XLOC_000006', array([0., 0., 0., ..., 0., 0., 0.])],
       ...,
       ['XLOC_030014', array([0., 0., 0., ..., 0., 0., 0.])],
       ['XLOC_030017', array([0., 0., 0., ..., 0., 0., 0.])],
       ['XLOC_030018', array([0., 0., 0., ..., 0., 0., 0.])]],
      dtype=object)

In [18]:
def split_data(X, y):
    # Split the dataset into training, validation, and test sets
    X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.666, random_state=42, stratify=y)
    X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

    return X_train, X_val, X_test, y_train, y_val, y_test

In [19]:
sample_np

array([['XLOC_000001', 0.0, 'XLOC_000001', 0.0, 0],
       ['XLOC_000001', 0.0, 'XLOC_000003', 0.0, 0],
       ['XLOC_000001', 0.0, 'XLOC_000006', 0.0888452, 0],
       ...,
       ['XLOC_000008', 26.7934, 'XLOC_014459', 0.0, 1],
       ['XLOC_000008', 26.7934, 'XLOC_014460', 0.0, 1],
       ['XLOC_000008', 26.7934, 'XLOC_014461', 0.0, 1]], dtype=object)

In [20]:
# Create X and y matrix
X_sample_np = sample_np[:, [0, 2]]
y_sample_np = sample_np[:, 4]

In [21]:
X_sample_np

array([['XLOC_000001', 'XLOC_000001'],
       ['XLOC_000001', 'XLOC_000003'],
       ['XLOC_000001', 'XLOC_000006'],
       ...,
       ['XLOC_000008', 'XLOC_014459'],
       ['XLOC_000008', 'XLOC_014460'],
       ['XLOC_000008', 'XLOC_014461']], dtype=object)

In [22]:
print(len(y_sample_np))
y_sample_np

100000


array([0, 0, 0, ..., 1, 1, 1], dtype=object)

In [23]:
X_train, X_val, X_test, y_train, y_val, y_test = split_data(X_sample_np, y_sample_np)
print(X_train.shape)
print(X_val.shape)
print(X_test.shape)
print(y_train.shape)
print(y_val.shape)
print(y_test.shape)

(33400, 2)
(33300, 2)
(33300, 2)
(33400,)
(33300,)
(33300,)


In [ ]:
def generate_batch(X, y, gene_features, batch_size = 32):
    indices = np.random.permutation(len(X))
    
    for i in range(0, len(X), batch_size):
        batch_indices = indices[i:i + batch_size]
        X_batch = np.empty((0, 40000), float)
        y_batch = np.empty((0, 1), float)
        
        for i in batch_indices:
            # Get feature 1
            gene_1 = X[i, 0]
            feature_1 = gene_features[gene_features[:, 0] == gene_1, 1]
            flattened_feature_1 = np.concatenate(feature_1, axis=None)
            
            # Get feature 2
            gene_2 = X[i, 1]
            feature_2 = gene_features[gene_features[:, 0] == gene_2, 1]
            flattened_feature_2 = np.concatenate(feature_2, axis=None)
            
            # Get the label
            label = y[i]
            
            # Append to the appropriate matrix
            row = np.concatenate((flattened_feature_1, flattened_feature_2), axis=0)
            X_batch = np.vstack((X_batch, row))
            y_batch = np.vstack((y_batch, label))

        yield torch.FloatTensor(X_batch).to(device), torch.FloatTensor(y_batch).to(device)

In [ ]:
# Manual batch generation
for X_batch, y_batch in generate_batch(X_train, y_train, data_np, batch_size=128):
    print(X_batch.shape, y_batch.shape)

In [ ]:
class BinaryClassifier(nn.Module):
    def __init__(self, input_size, hidden1_size=64, hidden2_size=32, output_size=1):
        super(BinaryClassifier, self).__init__()

        self.network = nn.Sequential(
            nn.Linear(input_size, hidden1_size),
            nn.ReLU(),
            nn.Linear(hidden1_size, hidden2_size),
            nn.ReLU(),
            nn.Linear(hidden2_size, output_size),
            nn.Sigmoid()
        )

    def forward(self, x):
        # x = x.squeeze(1)
        return self.network(x)

    def predict(self, x):
        with torch.no_grad():
            outputs = self.forward(x)
            return (outputs >= 0.5).long()

In [ ]:
# Initialize model
model = BinaryClassifier(input_size=40000).to(device)

# Define loss function for binary classification
criterion = nn.BCELoss()  # Binary Cross Entropy Loss

# Training loop example
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

num_epochs = 100

# Training loop
for epoch in range(num_epochs):
    model.train()
    
    total_loss = 0
    correct = 0
    total = 0
    
    # Manual batch generation
    for X_batch, y_batch in generate_batch(X_train, y_train, data_np, batch_size=32):
        # Forward pass
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        
        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # Calculate accuracy
        predictions = (outputs >= 0.5).long()
        correct += (predictions == y_batch).sum().item()
        total += y_batch.size(0)
        total_loss += loss.item()
    
    # Print epoch statistics
    avg_loss = total_loss / len(X_train)
    accuracy = correct / total
    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}, Accuracy: {accuracy:.4f}')